# 05 · Sequence models

Trains recurrent models on the 100–2000 Hz spectrograms produced by notebook 03. Each spectrogram is
treated as a time sequence, one step per column with the frequency bins as that step's features.
Predictions for a recording's clips are averaged before scoring.

### Architectures

| name | description | parameters |
|---|---|---|
| `bilstm` | bidirectional LSTM, hidden 64 per direction | 225k |
| `bigru` | bidirectional GRU, hidden 64 per direction | 175k |

A GRU has three gates to the LSTM's four and no separate cell state, so it carries roughly 25% fewer
parameters at the same hidden size.

Folds, seeds, augmentation, loss, early stopping, roll-up, metrics and baselines are identical to
notebook 04. Only the model body and learning rate differ.

### Before running

Notebook 03 must have produced `merged_features.csv` and at least one spectrogram configuration.
This notebook can run before or after notebook 04; whichever runs first creates the shared fold
files, and the other re-reads them.

In [ ]:
!pip install -q torch numpy pandas scipy matplotlib

Set `CONFIGS = None` to use every spectrogram configuration in `prepared/`.

In [ ]:
import os

BASE = f"/scratch/{os.environ.get('USER','username')}/capstone"

CONFIGS        = ["nfft1024_hop256"]
ARCHS_TO_RUN   = ["bilstm", "bigru"]
TARGETS_TO_RUN = ["shannon_rare_no_anchoa", "richness_rare_no_anchoa"]
SEEDS_TO_RUN   = [0, 1, 2, 3, 4]

PERMUTE      = False
PERMUTE_SEED = 0

K_FOLDS  = 5
EPOCHS   = 40
BATCH    = 64
DROPOUT  = 0.5
WEIGHT_DECAY = 1e-2

NORM = "group"

USE_SWA   = True
SWA_START = 6

VAL_FRAC  = 0.25
PATIENCE  = 10
WARMUP    = 5
MIN_DELTA = 0.01
SMOOTH    = 3

LR_BY_ARCH = {"bilstm": 1e-4, "bigru": 1e-4}

TIME_POOL   = 4
LSTM_HIDDEN = 64
ATTN_POOL   = True

DATA_TO_GPU = True

FEATURES_CSV = os.path.join(BASE, "merged_features.csv")

PREPARED = os.path.join(BASE, "prepared")
RUNS     = os.path.join(BASE, "runs"); os.makedirs(RUNS, exist_ok=True)
FIG_DIR  = os.path.join(BASE, "figures", "training"); os.makedirs(FIG_DIR, exist_ok=True)
LABELS   = ["richness_rare_no_anchoa", "shannon_rare_no_anchoa"]

print("project root:", os.path.basename(BASE.rstrip("/")) + "/")
print(f"archs = {ARCHS_TO_RUN} | targets = {TARGETS_TO_RUN} | seeds = {SEEDS_TO_RUN} | K = {K_FOLDS}"
      + (" | PERMUTED" if PERMUTE else ""))

if USE_SWA and EPOCHS < SWA_START + 2:
    raise SystemExit(f"EPOCHS ({EPOCHS}) must exceed SWA_START ({SWA_START}) by at least 2, "
                     f"or weight averaging never engages.")
if EPOCHS <= WARMUP:
    raise SystemExit(f"EPOCHS ({EPOCHS}) must exceed WARMUP ({WARMUP}), "
                     f"or no epoch is eligible to be restored.")


project root: capstone/
archs = ['bilstm', 'bigru'] | targets = ['shannon_rare_no_anchoa', 'richness_rare_no_anchoa'] | seeds = [0, 1, 2, 3, 4] | K = 5


Configurations must contain `spectrograms.npy`, `groups.npy` and `meta.json`. Incomplete
directories are listed with the reason.

In [ ]:
import json, glob
import numpy as np, pandas as pd

def discover_configs(prepared_dir, want_labels=()):
    usable, skipped = [], []
    for d in sorted(glob.glob(os.path.join(prepared_dir, "*"))):
        if not os.path.isdir(d) or os.path.basename(d).startswith("_"):
            continue
        name = os.path.basename(d)
        need = ["spectrograms.npy", "groups.npy", "meta.json"]
        missing = [f for f in need if not os.path.exists(os.path.join(d, f))]
        if missing:
            skipped.append((name, "missing " + ", ".join(missing))); continue
        try:
            m = json.load(open(os.path.join(d, "meta.json")))
        except Exception as e:
            skipped.append((name, f"unreadable meta.json ({e})")); continue
        usable.append(dict(config=name, path=d, n_fft=m.get("N_FFT"), hop=m.get("HOP"),
                           n_freq=m.get("n_freq"), n_time=m.get("n_time"),
                           n_clips=m.get("n_clips"), n_rec=m.get("n_recordings"),
                           win_ms=m.get("win_ms")))
    usable.sort(key=lambda c: (c["n_fft"] or 0, c["hop"] or 0))
    return usable, skipped

if not os.path.exists(FEATURES_CSV):
    raise SystemExit(f"{FEATURES_CSV} not found. Run 03_make_spectrograms.ipynb first.")
FEATURES = pd.read_csv(FEATURES_CSV,
                       usecols=lambda c: c in ({"recording_key"} | set(LABELS)))
missing_t = [t for t in LABELS if t not in FEATURES.columns]
if missing_t:
    raise SystemExit(f"{FEATURES_CSV} has no column(s) {missing_t}")
PER_REC = FEATURES.groupby("recording_key")[LABELS].first()
print(f"targets from {os.path.basename(FEATURES_CSV)}: {PER_REC.shape[0]} recordings | {LABELS}")

found, skipped = discover_configs(PREPARED)
if not found:
    raise SystemExit(f"No complete builds in {PREPARED}. Run 03_make_spectrograms.ipynb first.")
if CONFIGS is not None:
    names = {c["config"] for c in found}
    unknown = [c for c in CONFIGS if c not in names]
    if unknown:
        raise SystemExit(f"Config(s) not found or incomplete: {unknown}\nAvailable: {sorted(names)}")
    found = [c for c in found if c["config"] in CONFIGS]

print(f"{len(found)} config(s):")
print(pd.DataFrame(found)[["config","n_fft","hop","win_ms","n_freq","n_time","n_clips","n_rec"]]
      .to_string(index=False))
if skipped:
    print("\nskipped:")
    for n, w in skipped: print(f"   {n}: {w}")

targets from merged_features.csv: 72 recordings | ['richness_rare_no_anchoa', 'shannon_rare_no_anchoa']
1 config(s):
         config  n_fft  hop     win_ms  n_freq  n_time  n_clips  n_rec
nfft1024_hop256   1024  256 170.666667     324     235     2670     72


Device selection.

In [4]:
import torch, torch.nn as nn, random
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


**Loading.** `load_config` swaps in a build and re-derives everything shape-dependent; `load_target`
swaps in a label.

In [ ]:
spec = groups = meta = Y = None
rec_ids = rec_label = label_info = None
VALID_RECS = None
TARGET_SD = None
N = F = T = 0
CONFIG_NAME = PREP = TARGET = None
FREQ_MASK = TIME_MASK = 0

FREQ_MASK_FRAC = 0.20
TIME_MASK_FRAC = 0.15

def load_config(config_name, verbose=True):
    global spec, groups, meta, rec_ids, N, F, T
    global CONFIG_NAME, PREP, label_info, FREQ_MASK, TIME_MASK
    CONFIG_NAME = config_name
    PREP = os.path.join(PREPARED, config_name)
    try:
        arr = np.load(os.path.join(PREP, "spectrograms.npy"))
    except MemoryError:
        print("    (not enough RAM -- falling back to memory-mapped reads, which will be slower)")
        spec = np.load(os.path.join(PREP, "spectrograms.npy"), mmap_mode="r")
    else:
        spec = torch.from_numpy(arr)
        where = "RAM"
        if DATA_TO_GPU and device.type == "cuda":
            free, _ = torch.cuda.mem_get_info()
            if arr.nbytes < 0.4 * free:
                spec = spec.to(device); where = "VRAM"
            else:
                print(f"    (array {arr.nbytes/1e9:.2f} GB vs {free/1e9:.2f} GB free -- keeping in RAM)")
        print(f"    spectrograms held in {where}")
    groups = np.load(os.path.join(PREP, "groups.npy"))
    meta   = json.load(open(os.path.join(PREP, "meta.json")))
    label_info = {}
    N, F, T = tuple(spec.shape)
    rec_ids = np.unique(groups)
    FREQ_MASK = max(1, int(round(FREQ_MASK_FRAC * F)))
    TIME_MASK = max(1, int(round(TIME_MASK_FRAC * T)))
    if verbose:
        nb = spec.nbytes if hasattr(spec, "nbytes") else spec.element_size()*spec.nelement()
        print(f"config: {meta['config_name']} | {tuple(spec.shape)} ({nb/1e9:.2f} GB) | "
              f"{N} clips, {len(rec_ids)} recordings")

def load_target(target, verbose=True, permute=False, permute_seed=0):
    global Y, rec_label, TARGET, TARGET_SD, VALID_RECS
    if target not in PER_REC.columns:
        raise FileNotFoundError(f"{target} not in {os.path.basename(FEATURES_CSV)}")
    TARGET = target
    order = meta["rec_order"]
    absent = [r for r in order if r not in PER_REC.index]
    if absent:
        raise SystemExit(f"in the spectrogram cache but not in the CSV: {absent[:5]}")
    vals = PER_REC.loc[order, target].to_numpy(dtype=np.float32)
    if permute:
        vals = vals.copy(); np.random.default_rng(permute_seed).shuffle(vals)
    Y = vals[groups]
    rec_label = {int(r): float(vals[int(r)]) for r in rec_ids}
    VALID_RECS = np.array([int(r) for r in rec_ids if np.isfinite(rec_label[int(r)])])
    n_drop = len(rec_ids) - len(VALID_RECS)
    TARGET_SD = float(np.std([rec_label[int(r)] for r in VALID_RECS]))
    if verbose:
        v = np.array([rec_label[int(r)] for r in VALID_RECS])
        print(f"target '{target}'{' [PERMUTED]' if permute else ''}: "
              f"{v.min():.3f}-{v.max():.3f} | mean {v.mean():.3f} | sd {v.std():.3f}"
              + (f" | {n_drop} dropped for a missing target" if n_drop else ""))

load_config(found[0]["config"])
load_target(TARGETS_TO_RUN[0])

    spectrograms held in VRAM
config: nfft1024_hop256 | (2670, 324, 235) (0.81 GB) | 2670 clips, 72 recordings
target 'shannon_rare_no_anchoa': 0.699-2.171 | mean 1.334 | sd 0.372


Fold assignments are written once per seed and re-read by every subsequent run.

In [ ]:
def load_or_make_folds(seed, k=K_FOLDS, verbose=True):
    p = os.path.join(RUNS, f"folds_k{k}_seed{seed}.json")
    rec_order = meta["rec_order"]
    ids = VALID_RECS if VALID_RECS is not None else rec_ids
    if os.path.exists(p):
        d = json.load(open(p))
        missing = [rec_order[int(r)] for r in ids if rec_order[int(r)] not in d["folds"]]
        if missing:
            raise SystemExit(f"{p} does not cover: {missing[:3]} ... delete it to regenerate.")
        return np.array([d["folds"][rec_order[int(r)]] for r in ids], dtype=int)
    rng = np.random.default_rng(seed)
    ids = np.asarray(ids).copy(); rng.shuffle(ids)
    assign = {}
    for f, part in enumerate(np.array_split(ids, k)):
        for r in part:
            assign[rec_order[int(r)]] = f
    json.dump(dict(k=k, seed=seed, n_recordings=len(rec_order), folds=assign),
              open(p, "w"), indent=2)
    if verbose:
        print(f"    wrote shared fold file -> {os.path.basename(p)}")
    ids2 = VALID_RECS if VALID_RECS is not None else rec_ids
    return np.array([assign[rec_order[int(r)]] for r in ids2], dtype=int)

Augmentation applies Gaussian noise and SpecAugment-style masking. Mask widths are derived from
the spectrogram shape when a configuration is loaded.

In [ ]:
AUGMENT = True
NOISE   = 0.15
N_MASKS = 2

def augment(x):
    if NOISE > 0: x = x + NOISE * torch.randn_like(x)
    Fd, Td = x.shape[-2], x.shape[-1]
    for _ in range(N_MASKS):
        if FREQ_MASK > 0:
            f = random.randint(0, min(FREQ_MASK, Fd)); f0 = random.randint(0, max(0, Fd - f))
            x[..., f0:f0+f, :] = 0.0
        if TIME_MASK > 0:
            t = random.randint(0, min(TIME_MASK, Td)); t0 = random.randint(0, max(0, Td - t))
            x[..., :, t0:t0+t] = 0.0
    return x

def batches(idx, bs, shuffle, s_mu, s_sd, t_mu, t_sd):
    idx = np.asarray(idx)
    order = np.random.permutation(len(idx)) if shuffle else np.arange(len(idx))
    sel_all = idx[order]
    for k in range(0, len(sel_all), bs):
        sel = sel_all[k:k+bs]
        if torch.is_tensor(spec):
            x = spec.index_select(0, torch.as_tensor(sel, device=spec.device))
            x = x.to(device, non_blocking=True).float()
        else:
            sel = np.sort(sel)
            x = torch.from_numpy(np.asarray(spec[sel], dtype=np.float32)).to(device)
        x = ((x - s_mu) / s_sd).unsqueeze(1)         # (B, 1, F, T)
        y = torch.from_numpy(((Y[sel] - t_mu) / t_sd).astype(np.float32)).to(device)
        yield x, y, sel

Both models average-pool the time axis, run a bidirectional recurrent layer over the remaining steps,
aggregate across time and pass the result to the same head the image models use. The input width is
the number of frequency bins, taken from the loaded configuration.

With `ATTN_POOL` set, aggregation is a learned weighted average rather than a plain mean: a small
network scores each timestep and the scores are normalised into weights.

In [ ]:
class AttnPool(nn.Module):
    def __init__(self, d, hidden=64):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(d, hidden), nn.Tanh(), nn.Linear(hidden, 1))
    def forward(self, h):
        w = torch.softmax(self.score(h).squeeze(-1), dim=1)
        return (w.unsqueeze(-1) * h).sum(1), w

class RecurrentNet(nn.Module):
    def __init__(self, n_freq, cell="lstm", p=0.5, pool=TIME_POOL, hid=LSTM_HIDDEN):
        super().__init__()
        self.pool = nn.AvgPool1d(pool)
        rnn = nn.LSTM if cell == "lstm" else nn.GRU
        self.rnn  = rnn(n_freq, hid, batch_first=True, bidirectional=True)
        self.attn = AttnPool(hid*2) if ATTN_POOL else None
        self.norm = nn.LayerNorm(hid*2); self.drop = nn.Dropout(p)
        self.head = nn.Sequential(nn.Linear(hid*2,128), nn.ReLU(), nn.Dropout(p), nn.Linear(128,1))
    def forward(self, x):
        xf = self.pool(x.squeeze(1)).transpose(1, 2)
        out, _ = self.rnn(xf)
        z = self.attn(out)[0] if self.attn is not None else out.mean(1)
        return self.head(self.drop(self.norm(z))).squeeze(-1)

class BiLSTM(RecurrentNet):
    def __init__(self, p=0.5): super().__init__(F, "lstm", p)

class BiGRU(RecurrentNet):
    def __init__(self, p=0.5): super().__init__(F, "gru", p)

ARCHS = {"bilstm": BiLSTM, "bigru": BiGRU}

def build_net(arch): return ARCHS[arch](DROPOUT).to(device)
def n_params(arch):  return sum(p.numel() for p in ARCHS[arch](DROPOUT).parameters())

print(f"input: {F} frequency bins per step | {T} frames -> {T // TIME_POOL} steps "
      f"after pooling by {TIME_POOL}")
print("parameter counts:")
for a in ARCHS:
    print(f"  {a:9s} {n_params(a):>9,d}")
with torch.no_grad():
    xz = torch.zeros(2, 1, F, T, device=device)
    for a in ARCHS:
        print(f"  {a:9s} forward OK -> {tuple(build_net(a)(xz).shape)}")

input: 324 frequency bins per step | 235 frames -> 58 steps after pooling by 4
parameter counts:
  bilstm      224,898
  bigru       174,978
  bilstm    forward OK -> (2,)
  bigru     forward OK -> (2,)


## Cross-validation

Each fold trains a new model on the remaining folds and predicts the held-out recordings. A
`VAL_FRAC` slice of the training recordings is reserved for early stopping and never overlaps the
test fold.

Training loss is Smooth L1 in standardized units; validation error is MAE in the target's own units.
`val_mae_std` is the validation MAE divided by the fold's target standard deviation, so both can be
plotted on one axis.

Two baselines are computed: predicting the training mean, and predicting the training median. The
mean is the reference for R²; the median is the harder reference for MAE, so `beats_base` requires
beating both.

In [ ]:
def r2_score(y, p, ref_sd=None):
    y = np.asarray(y, float); p = np.asarray(p, float)
    if y.size < 2: return float("nan")
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    if ss_tot <= 0: return float("nan")
    if ref_sd is not None and ref_sd > 0 and float(np.std(y)) < 0.05 * ref_sd:
        return float("nan")
    return 1.0 - float(np.sum((y - p) ** 2)) / ss_tot

def metrics(true, pred, ref_sd=None):
    true = np.asarray(true, float); pred = np.asarray(pred, float)
    return dict(mae=float(np.abs(pred-true).mean()),
                rmse=float(np.sqrt(np.mean((pred-true)**2))),
                r2=r2_score(true, pred, ref_sd=ref_sd))

_STATS_CACHE = {}

def spec_mean_std(idx, key=None, chunk=512):
    if key is not None and key in _STATS_CACHE:
        return _STATS_CACHE[key]
    idx = np.asarray(idx); tot = sq = 0.0; cnt = 0
    for i in range(0, len(idx), chunk):
        sel = idx[i:i+chunk]
        if torch.is_tensor(spec):
            blk = spec.index_select(0, torch.as_tensor(sel, device=spec.device)).double()
            tot += float(blk.sum()); sq += float((blk**2).sum()); cnt += blk.numel()
        else:
            blk = np.asarray(spec[np.sort(sel)], dtype=np.float64)
            tot += blk.sum(); sq += (blk**2).sum(); cnt += blk.size
    mu = tot/cnt
    out = (float(mu), float(np.sqrt(max(sq/cnt - mu*mu, 0.0)) + 1e-6))
    if key is not None: _STATS_CACHE[key] = out
    return out

def train_fold(arch, fit_idx, val_idx, te_idx, seed, stats_key=None):
    s_mu, s_sd = spec_mean_std(fit_idx, key=stats_key)
    t_mu, t_sd = float(Y[fit_idx].mean()), float(Y[fit_idx].std() + 1e-6)
    mk = lambda ix, sh: (lambda: batches(ix, BATCH, sh, s_mu, s_sd, t_mu, t_sd))
    tr_dl, va_dl, te_dl = mk(fit_idx, True), mk(val_idx, False), mk(te_idx, False)

    torch.manual_seed(seed); net = build_net(arch)
    opt  = torch.optim.AdamW(net.parameters(), lr=LR_BY_ARCH.get(arch, 3e-4),
                             weight_decay=WEIGHT_DECAY)
    crit = nn.SmoothL1Loss()
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, "min", factor=0.5, patience=3)

    def predict(fn):
        net.eval(); P, I = [], []
        with torch.no_grad():
            for x, y, sel in fn():
                P.append(net(x).cpu().numpy()); I.append(sel)
        return np.concatenate(P)*t_sd + t_mu, np.concatenate(I)

    hist = {"epoch": [], "train_loss": [], "val_mae": [], "val_mae_smooth": [],
            "val_mae_std": [], "lr": []}
    best = float("inf"); bad = 0; best_state = None; best_ep = 0; stopped_early = False
    min_delta_abs = MIN_DELTA * t_sd
    swa_state, swa_n = None, 0

    for ep in range(1, EPOCHS + 1):
        net.train(); run = 0.0; nb = 0
        for x, y, _ in tr_dl():
            if AUGMENT: x = augment(x)
            opt.zero_grad(); loss = crit(net(x), y); loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
            run += float(loss.item()); nb += 1
        Pv, Iv = predict(va_dl); v = float(np.abs(Pv - Y[Iv]).mean())
        sched.step(v)
        hist["epoch"].append(ep); hist["train_loss"].append(run/max(1,nb))
        hist["val_mae"].append(v)
        vs = float(np.mean(hist["val_mae"][-SMOOTH:]))
        hist["val_mae_smooth"].append(vs)
        hist["val_mae_std"].append(vs / t_sd)
        hist["lr"].append(float(opt.param_groups[0]["lr"]))

        if USE_SWA and ep >= SWA_START:
            sd_now = net.state_dict()
            if swa_state is None:
                swa_state = {k: v.detach().clone().float() for k, v in sd_now.items()}
                swa_n = 1
            else:
                swa_n += 1
                for k, v in sd_now.items():
                    if swa_state[k].is_floating_point():
                        swa_state[k] += (v.detach().float() - swa_state[k]) / swa_n
                    else:
                        swa_state[k] = v.detach().clone()

        if ep < WARMUP:
            continue
        if vs < best - min_delta_abs:
            best, bad, best_ep = vs, 0, ep
            best_state = {k: t.clone() for k, t in net.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: stopped_early = True; break

    used = "best_epoch"
    if USE_SWA and swa_state is not None and swa_n > 1:
        net.load_state_dict({k: v.to(dtype=net.state_dict()[k].dtype)
                             for k, v in swa_state.items()})
        if NORM == "batch":
            for mod in net.modules():
                if isinstance(mod, nn.BatchNorm2d):
                    mod.reset_running_stats(); mod.momentum = None
            net.train()
            with torch.no_grad():
                for x, _, _ in tr_dl():
                    net(x)
        used = f"swa({swa_n} epochs)"
    elif best_state is not None:
        net.load_state_dict(best_state)
    pred_te, idx_te = predict(te_dl)
    stop = dict(epochs_run=hist["epoch"][-1], best_epoch=best_ep, epoch_cap=EPOCHS,
                early_stopped=bool(stopped_early), patience=PATIENCE,
                warmup=WARMUP, min_delta_frac=MIN_DELTA, smooth=SMOOTH,
                weights_used=used, swa_epochs=int(swa_n),
                val_recordings=int(len(np.unique(groups[val_idx]))),
                best_val_mae=float(best), reason=("early stop" if stopped_early else "epoch cap"))
    fit_vals = [rec_label[int(r)] for r in np.unique(groups[fit_idx])]
    return pred_te, idx_te, float(np.mean(fit_vals)), float(np.median(fit_vals)), hist, stop

def run_cv(arch, seed, verbose=True):
    fold_of = load_or_make_folds(seed, K_FOLDS, verbose=verbose)
    scored = VALID_RECS if VALID_RECS is not None else rec_ids
    clip_pred = np.full(N, np.nan)
    rec_pred, base_mean, base_med = {}, {}, {}
    per_fold, histories, stops = [], [], []

    for f in range(K_FOLDS):
        te_recs = scored[fold_of == f]
        tr_recs = scored[fold_of != f]
        rf = np.random.default_rng(1000 + f); tr_recs = tr_recs.copy(); rf.shuffle(tr_recs)
        n_val = max(2, int(round(VAL_FRAC * len(tr_recs))))
        val_recs, fit_recs = tr_recs[:n_val], tr_recs[n_val:]

        fit_idx = np.where(np.isin(groups, fit_recs))[0]
        val_idx = np.where(np.isin(groups, val_recs))[0]
        te_idx  = np.where(np.isin(groups, te_recs))[0]

        preds, idx_te, mu, med, hist, stop = train_fold(
            arch, fit_idx, val_idx, te_idx, seed*100 + f,
            stats_key=(CONFIG_NAME, seed, f))
        clip_pred[idx_te] = preds
        for r in te_recs:
            ci = np.where(groups == r)[0]
            rec_pred[int(r)]  = float(np.nanmean(clip_pred[ci]))
            base_mean[int(r)] = mu; base_med[int(r)] = med

        m = metrics([rec_label[int(r)] for r in te_recs],
                    [rec_pred[int(r)] for r in te_recs], ref_sd=TARGET_SD)
        per_fold.append(m); histories.append(hist); stops.append(stop)
        if verbose:
            tag = "EARLY STOP" if stop["early_stopped"] else "EPOCH CAP "
            print(f"    fold {f+1}/{K_FOLDS}: test={len(te_recs):2d} | "
                  f"ran {stop['epochs_run']:2d}/{EPOCHS} | best@{stop['best_epoch']:2d} | {tag} | "
                  f"MAE={m['mae']:.4f} RMSE={m['rmse']:.4f} R2={m['r2']:.3f}")

    recs  = list(rec_pred)
    truth = [rec_label[r] for r in recs]
    rec_m  = metrics(truth, [rec_pred[r]  for r in recs], ref_sd=TARGET_SD)
    b_mean = metrics(truth, [base_mean[r] for r in recs], ref_sd=TARGET_SD)
    b_med  = metrics(truth, [base_med[r]  for r in recs], ref_sd=TARGET_SD)

    if verbose:
        ne = sum(s["early_stopped"] for s in stops)
        eps = [s["epochs_run"] for s in stops]; bst = [s["best_epoch"] for s in stops]
        print(f"    stopping: {ne}/{K_FOLDS} early-stopped, {K_FOLDS-ne}/{K_FOLDS} hit the cap | "
              f"epochs {min(eps)}/{int(np.median(eps))}/{max(eps)} | "
              f"best {min(bst)}/{int(np.median(bst))}/{max(bst)}")
        print(f"    validation split: {stops[0]['val_recordings']} recordings per fold | "
              f"weights: {stops[0]['weights_used']}")
        if ne < K_FOLDS:
            print(f"    NOTE: {K_FOLDS-ne} fold(s) still improving at epoch {EPOCHS}.")
        if not USE_SWA:
            if min(bst) < WARMUP + 2:
                print(f"    NOTE: a fold restored weights from epoch {min(bst)}, barely past warmup "
                      f"-- the validation signal may be too noisy to select on.")
            if max(bst) - min(bst) > 12:
                print(f"    NOTE: best epoch ranges {min(bst)}-{max(bst)} across folds. A tight "
                      f"cluster would suggest a real optimum; this much scatter suggests there is "
                      f"little to learn beyond the mean.")

    return dict(per_fold=per_fold, rec=rec_m, base_mean=b_mean, base_median=b_med,
                n_rec=len(recs), histories=histories, stops=stops,
                rec_pred=rec_pred, rec_true={int(r): rec_label[int(r)] for r in recs},
                fold_of={int(r): int(fold_of[i]) for i, r in enumerate(scored)})

Loss curves for all folds of one run. Solid lines are training loss, dashed are validation error
on the same scale, and the star marks the restored epoch.

In [10]:
def plot_loss_curves(res, model, target, config_name, k_folds, seed, out_dir=None, show=False):
    hists, stops = res["histories"], res["stops"]
    cmap = plt.get_cmap("tab10")
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for i, (h, s) in enumerate(zip(hists, stops)):
        c = cmap(i % 10)
        ax.plot(h["epoch"], h["train_loss"],  color=c, lw=1.6, ls="-",  label=f"fold {i+1} train")
        ax.plot(h["epoch"], h["val_mae_std"], color=c, lw=1.3, ls="--", label=f"fold {i+1} val")
        be = s["best_epoch"]
        ax.plot([be], [h["val_mae_std"][be-1]], marker="*", ms=13, color=c, mec="black", mew=0.6)
    ne = sum(s["early_stopped"] for s in stops)
    ax.axvline(EPOCHS, color="0.4", lw=1, ls=":")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss / error (standardized units)")
    ax.set_title(f"{model} · {target} · {config_name} · seed {seed}\n"
                 f"solid = train, dashed = validation, star = restored best epoch · "
                 f"{ne}/{k_folds} folds early-stopped", fontsize=11)
    ax.grid(alpha=0.3); ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
    if out_dir:
        tag = "_perm" if PERMUTE else ""
        fig.savefig(os.path.join(out_dir,
                    f"{model}_{target}_{config_name}_cv{k_folds}_seed{seed}{tag}_losscurves.png"),
                    dpi=150, bbox_inches="tight")
    if show: plt.show()
    plt.close(fig)

## Run

Iterates over every configuration, architecture, target and seed. Each run writes a results file, an
out-of-fold prediction CSV and a loss-curve figure to `runs/`. A failure in one run is reported and
the remainder continue.

The out-of-fold CSV holds one row per recording, each prediction made by the fold model that did not
see that recording. Use these, not in-sample predictions, when combining with other models.

In [11]:
import time, traceback

rows, t_start = [], time.time()
n_jobs = len(found) * len(ARCHS_TO_RUN) * len(TARGETS_TO_RUN) * len(SEEDS_TO_RUN)
job = 0
print(f"{n_jobs} run(s) queued" + (" | PERMUTED" if PERMUTE else "") + "\n")

for cfg in found:
    load_config(cfg["config"], verbose=False)
    for arch in ARCHS_TO_RUN:
        for target in TARGETS_TO_RUN:
            for seed in SEEDS_TO_RUN:
                job += 1
                print(f"\n[{job}/{n_jobs}] {arch} -> {target} | {cfg['config']} | seed={seed}",
                      flush=True)
                try:
                    load_target(target, verbose=False, permute=PERMUTE,
                                permute_seed=PERMUTE_SEED + seed)
                except FileNotFoundError as e:
                    print(f"    SKIP: missing {os.path.basename(str(e))}"); continue
                t0 = time.time()
                try:
                    res = run_cv(arch, seed)
                except Exception:
                    print("    FAILED:"); traceback.print_exc(limit=2); continue
                mins = (time.time() - t0) / 60

                h, bmu, bmd = res["rec"], res["base_mean"], res["base_median"]
                pf = pd.DataFrame(res["per_fold"])
                ne = sum(s["early_stopped"] for s in res["stops"])
                hard_mae = min(bmu["mae"], bmd["mae"])
                print(f"    POOLED: MAE={h['mae']:.4f} (mean-base {bmu['mae']:.4f}, "
                      f"median-base {bmd['mae']:.4f}) | RMSE={h['rmse']:.4f} "
                      f"(base {bmu['rmse']:.4f}) | R2={h['r2']:.3f} | {mins:.1f} min")
                if h["mae"] < bmu["mae"] and h["mae"] >= bmd["mae"]:
                    print("    NOTE: beats the mean-baseline on MAE but not the median-baseline -- "
                          "consistent with shrinking toward the centre, not signal.")

                plot_loss_curves(res, arch, target, cfg["config"], K_FOLDS, seed, out_dir=FIG_DIR)

                tag  = "_perm" if PERMUTE else ""
                stem = f"{arch}_{target}_{cfg['config']}_cv{K_FOLDS}_seed{seed}{tag}"
                json.dump({"model": arch, "family": "sequence", "target": target,
                           "config_name": cfg["config"], "n_params": n_params(arch),
                           "n_fft": meta["N_FFT"], "hop": meta["HOP"],
                           "n_freq": meta["n_freq"], "n_time": meta["n_time"],
                           "win_ms": meta.get("win_ms"),
                           "level": "recording", "k_folds": K_FOLDS, "seed": seed,
                           "permuted": bool(PERMUTE), "epochs_cap": EPOCHS, "patience": PATIENCE,
                           "val_frac": VAL_FRAC, "warmup": WARMUP,
                           "min_delta_frac": MIN_DELTA, "smooth": SMOOTH,
                           "dropout": DROPOUT, "weight_decay": WEIGHT_DECAY, "norm": NORM,
                           "use_swa": bool(USE_SWA), "swa_start": SWA_START,
                           "noise": NOISE, "n_masks": N_MASKS,
                           "freq_mask_frac": FREQ_MASK_FRAC, "time_mask_frac": TIME_MASK_FRAC,
                           "attn_pool": bool(ATTN_POOL), "time_pool": TIME_POOL,
                           "lr": LR_BY_ARCH.get(arch), "minutes": round(mins, 2),
                           "targets_from": os.path.basename(FEATURES_CSV),
                           "headline": h, "baseline": bmu, "baseline_median": bmd,
                           "per_fold": res["per_fold"], "stops": res["stops"],
                           "histories": res["histories"]},
                          open(os.path.join(RUNS, f"{stem}_results.json"), "w"), indent=2)

                rows.append(dict(model=arch, family="sequence", target=target, config=cfg["config"],
                                 n_params=n_params(arch), seed=seed, permuted=bool(PERMUTE),
                                 MAE=h["mae"], RMSE=h["rmse"], R2=h["r2"],
                                 MAE_mean_base=bmu["mae"], MAE_median_base=bmd["mae"],
                                 RMSE_base=bmu["rmse"],
                                 MAE_fold_mean=float(pf["mae"].mean()),
                                 MAE_fold_sd=float(pf["mae"].std()),
                                 RMSE_fold_mean=float(pf["rmse"].mean()),
                                 RMSE_fold_sd=float(pf["rmse"].std()),
                                 R2_fold_mean=float(pf["r2"].mean()),
                                 R2_fold_median=float(pf["r2"].median()),
                                 R2_fold_sd=float(pf["r2"].std()),
                                 folds_early_stopped=f"{ne}/{K_FOLDS}",
                                 median_epochs=int(np.median([s["epochs_run"] for s in res["stops"]])),
                                 beats_base=bool(h["mae"] < hard_mae), minutes=round(mins, 1)))

20 run(s) queued

    spectrograms held in VRAM

[1/20] bilstm -> shannon_rare_no_anchoa | nfft1024_hop256 | seed=0
    fold 1/5: test=15 | ran 15/40 | best@ 5 | EARLY STOP | MAE=0.2276 RMSE=0.3002 R2=0.253
    fold 2/5: test=15 | ran 15/40 | best@ 5 | EARLY STOP | MAE=0.2321 RMSE=0.3174 R2=0.218
    fold 3/5: test=14 | ran 15/40 | best@ 5 | EARLY STOP | MAE=0.2919 RMSE=0.3378 R2=0.208
    fold 4/5: test=14 | ran 28/40 | best@18 | EARLY STOP | MAE=0.2915 RMSE=0.3542 R2=0.121
    fold 5/5: test=14 | ran 27/40 | best@17 | EARLY STOP | MAE=0.2497 RMSE=0.3297 R2=0.086
    stopping: 5/5 early-stopped, 0/5 hit the cap | epochs 15/15/28 | best 5/5/18
    validation split: 14 recordings per fold | weights: swa(10 epochs)
    POOLED: MAE=0.2578 (mean-base 0.3078, median-base 0.3073) | RMSE=0.3279 (base 0.3796) | R2=0.222 | 0.2 min

[2/20] bilstm -> shannon_rare_no_anchoa | nfft1024_hop256 | seed=1
    fold 1/5: test=15 | ran 24/40 | best@14 | EARLY STOP | MAE=0.2732 RMSE=0.3284 R2=-0.426
    fo

Combined results across all runs.

In [12]:
if rows:
    s = pd.DataFrame(rows).sort_values(["target","model","seed"]).reset_index(drop=True)
    print(f"\n{'='*80}\nsequence models — {len(s)} run(s), {(time.time()-t_start)/60:.1f} min"
          + ("  [PERMUTED NULL]" if PERMUTE else "") + f"\n{'='*80}")
    show = ["model","target","n_params","seed","MAE","RMSE","R2","MAE_mean_base",
            "MAE_median_base","MAE_fold_mean","MAE_fold_sd","folds_early_stopped","minutes"]
    with pd.option_context("display.float_format", lambda v: f"{v:.4g}", "display.width", 250):
        print(s[show].to_string(index=False))

    print("\nmean over seeds (± seed-to-seed sd):")
    g = s.groupby(["target","model"]).agg(
        n=("seed","count"), params=("n_params","first"),
        R2=("R2","mean"), R2_sd=("R2","std"),
        MAE=("MAE","mean"), MAE_sd=("MAE","std"),
        RMSE=("RMSE","mean"), RMSE_sd=("RMSE","std")).reset_index()
    with pd.option_context("display.float_format", lambda v: f"{v:.4f}"):
        print(g.to_string(index=False))

    tag = "_perm" if PERMUTE else ""
    print("next: 06_results_summary.ipynb, then 07_gru_predictions.ipynb")
else:
    print("\nNo runs completed.")


sequence models — 20 run(s), 3.5 min
 model                  target  n_params  seed    MAE   RMSE       R2  MAE_mean_base  MAE_median_base  MAE_fold_mean  MAE_fold_sd folds_early_stopped  minutes
 bigru richness_rare_no_anchoa    174978     0  2.034  2.697    0.036          2.174            2.154          2.041         0.44                 5/5      0.2
 bigru richness_rare_no_anchoa    174978     1  1.926  2.462   0.1968           2.15            2.173          1.927       0.1389                 5/5      0.2
 bigru richness_rare_no_anchoa    174978     2  1.856  2.516   0.1609          2.142            2.173          1.845       0.5901                 5/5      0.2
 bigru richness_rare_no_anchoa    174978     3  2.029  2.691   0.0406          2.163            2.187          2.029       0.1682                 5/5      0.2
 bigru richness_rare_no_anchoa    174978     4  1.928  2.509    0.166          2.224            2.245          1.928       0.4256                 5/5      0.2
bilstm r